[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/11_nucleic_acid/11_nucleic_lab.ipynb)


# 11 — 핵산(DNA/RNA) 결합 단백질 실습

> 본문 [`11_nucleic_acid.md`](11_nucleic_acid.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 **여러분이 직접 돌린 결과**(`my_run/`)에서 계산합니다.
> **① 직접 설계 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 갑니다. 설계 셀은 NVIDIA GPU 전용이에요(CPU 폴백 없음) — Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU**.

## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `11_nucleic_acid/` 로 이동합니다. 로컬에서 `11_nucleic_acid/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "11_nucleic_acid"
PIP_PKGS = "pandas matplotlib gemmi py3Dmol"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update", quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y fonts-nanum", quiet=True)

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) 직접 돌려보기 — 내 결과 만들기

- 학습용 규모 `num_designs=8 --budget=4` (레퍼런스 결과는 num_designs=30)
- 소요 시간 실측 **585초**(최종 4개) — **24GB GPU · 가중치 캐시** 기준이에요. Colab T4 는 가속 커널이 꺼져 더 걸리고(T4 실측치 없음), 첫 실행은 가중치 ~6GB 다운로드가 더 붙어요.
- 건너뛰면 아래 분석이 커밋된 레퍼런스 결과로 이어집니다.

In [ ]:
SPEC, PROTOCOL = "example/denovo_zinc_finger_against_dna/zinc_finger.yaml", "protein-anything"
NUM_DESIGNS, BUDGET = 8, 4

import shutil
OUT = MY.resolve()

def _gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return shutil.which("nvidia-smi") is not None

if not _gpu():
    print("GPU 런타임이 아니라 설계 실행을 건너뜁니다 — 아래 분석은 레퍼런스 결과로 이어집니다.")
    print("  · 직접 돌리려면 Colab [런타임 → 런타임 유형 변경 → T4 GPU] 후 이 셀을 다시 실행하세요.")
else:
    SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세·타깃 CIF 가 들어 있는 BoltzGen 소스
    if not SRC.exists():
        _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')
    if not _have("boltzgen"):
        _run(f'"{sys.executable}" -m pip -q install -e "{SRC}"')
    try:
        _run(f'cd "{SRC}" && boltzgen run {SPEC} --output "{OUT}" --protocol {PROTOCOL} '
             f'--num_designs {NUM_DESIGNS} --budget {BUDGET}')
        print("\n내 결과 →", OUT / "final_ranked_designs")
    except Exception as e:
        print("\n설계 실행이 도중에 멈췄어요 —", type(e).__name__)
        print("  · 이 셀을 다시 실행하면 같은 --output 산출물을 재사용해 이어서 끝냅니다(실측 763초 → 재개 486초).")
        print("  · GPU 메모리가 부족했다면 NUM_DESIGNS 4, BUDGET 2 로 줄이세요.")

## 2) 핵산 타깃은 따로 선언하지 않는다 (본문 11.3)

방금 명령의 `--protocol` 은 `protein-anything` 이었어요. DNA 전용 프로토콜을 쓰지 않은 이유는, BoltzGen 이 CIF 의 잔기 코드(CCD)를 보고 **DNA·RNA 를 스스로 구분**하기 때문이에요. 그래서 가장 단순한 de novo DNA 바인더 명세(`denovo_zinc_finger_against_dna/vanilla_protein.yaml`)는 설계할 단백질 한 줄과 DNA 두 가닥을 `include` 하는 게 전부예요.

```yaml
entities:
  - protein: { id: G, sequence: 40..120 }   # DNA 에 붙을 새 단백질
  - file:
      path: zf.cif
      include:
        - chain: { id: C1 }   # DNA 가닥 1 — DNA 로 자동 인식
        - chain: { id: B1 }   # DNA 가닥 2
```

자동 분류의 결과는 산출물 CIF 의 `_entity_poly` 에 그대로 적혀 나와요.

In [ ]:
KIND = {"polypeptide(L)": "단백질", "polydeoxyribonucleotide": "DNA",
        "polyribonucleotide": "RNA"}
cif = find_one("final*designs/*.cif", "data/dna")
lines = pathlib.Path(cif).read_text().splitlines()
for l in lines:
    for key, ko in KIND.items():
        if key in l and l.split()[1] == key:
            seq = l.split()[-1]
            print(f"{ko:4s} {len(seq):4d} 잔기 | {seq[:44]}{'…' if len(seq) > 44 else ''}")
zn = [l for l in lines if l.startswith("HETATM") and l.split()[5] == "ZN"]
print(f"Zn 이온 {len(zn)}개 (비폴리머라 원자 단위로 셉니다)")

단백질 1사슬 + DNA 2가닥(각 36 nt) + Zn 6개가 한 복합체예요. 단백질을 뺀 타깃 쪽 토큰은 36+36+6 = 78 이고, 이 숫자는 뒤에서 CSV 의 `num_tokens − num_prot_tokens` 로 다시 만납니다.

## 3) zinc_finger.yaml — 고급 입력 기능 총동원 (본문 11.4)

앞 절의 `vanilla_protein.yaml` 이 "빈 종이에 새로 그리기"라면, 이번에 돌린 `zinc_finger.yaml` 은 "기존 zinc finger 를 뜯어고치기"예요. BoltzGen 에서 가장 복잡한 명세 중 하나고, Ch.02 의 고급 기능이 거의 다 나옵니다.

```yaml
entities:
  - file:
      path: zf.cif
      include: "all"                    # 전체 포함 후
      exclude:                          # 불필요한 구간 제외
        - chain: { id: A1, res_index: ..10,63..69,185.. }
      design_insertions:                # finger 사이 linker 삽입
        - insertion: { id: A1, res_index: 63, num_residues: 3..8 }
      structure_groups:                 # 구조를 숨겨 자유 재설계
        - group: { visibility: 0, id: "all" }
      design:                           # 재설계할 영역
        - chain: { id: A1, res_index: 11..184 }
      not_design:                       # Zn 배위 잔기 등 고정
        - chain: { id: A1, res_index: 11..20,29,33,39..48,57,61,72..81,90,94,100..109,118,122,129..138,147,151,157..166,175,179 }
      reset_res_index:                  # 번호 정리
        - chain: { id: A1 }
```

| 기능 | 무엇을 하나 | 왜 |
|------|------------|----|
| `exclude` | N/C 말단·특정 구간 제거 | 시스템 경량화 |
| `design_insertions` | finger 사이 linker 3~8잔기 삽입 | DNA 인식에 필요한 유연성 |
| `structure_groups: visibility 0` | 구조 숨김 → 자유 재설계 | 타깃 DNA 마다 최적 구조가 다름 |
| `design` / `not_design` | 재설계 영역 vs 고정 영역 | 기능 필수 잔기 보호 |
| `reset_res_index` | 잔기 번호 연속 정리 | exclude·insertion 후 정돈 |

말로만 보면 감이 안 오니 범위 문자열을 실제로 펼쳐 세어 보고, 그 결과가 결과 CSV 의 길이와 맞는지 대조해요.

In [ ]:
from boltzgen_viz import load_metrics, metrics_overview, compare_bars
import pandas as pd
DNA_CSV = str(find_one("final_designs_metrics_*.csv", "data/dna"))
dna = pd.read_csv(DNA_CSV).sort_values("final_rank")

CHAIN_LEN = 200          # zf.cif 의 A1 사슬 잔기 수
def expand(spec, lo=1, hi=CHAIN_LEN):
    """'..10,63..69,185..' 같은 res_index 표기를 잔기 번호 집합으로 펼친다."""
    out = set()
    for part in spec.split(","):
        if ".." in part:
            a, b = part.split("..")
            out |= set(range(int(a) if a else lo, (int(b) if b else hi) + 1))
        else:
            out.add(int(part))
    return out

EXCLUDE    = expand("..10,63..69,185..")
DESIGN     = expand("11..184")
NOT_DESIGN = expand("11..20,29,33,39..48,57,61,72..81,90,94,100..109,118,122,"
                    "129..138,147,151,157..166,175,179")
INSERTION  = (3, 8)

kept       = set(range(1, CHAIN_LEN + 1)) - EXCLUDE
fixed      = NOT_DESIGN & kept
designable = (DESIGN & kept) - NOT_DESIGN
print(f"원본 A1 {CHAIN_LEN}잔기 → exclude {len(EXCLUDE)}잔기 제거 → 남는 사슬 {len(kept)}잔기")
print(f"그중 not_design 으로 고정 {len(fixed)}잔기, 재설계 가능 {len(designable)}잔기")
print(f"+ design_insertions {INSERTION[0]}~{INSERTION[1]}잔기")
print(f"→ 예상 num_design {len(designable) + INSERTION[0]}~{len(designable) + INSERTION[1]},"
      f" 단백질 길이 {len(kept) + INSERTION[0]}~{len(kept) + INSERTION[1]}")
if {"num_design", "num_prot_tokens", "num_tokens"} <= set(dna.columns):
    print(f"\nCSV 실측 num_design {int(dna.num_design.min())}~{int(dna.num_design.max())},"
          f" num_prot_tokens {int(dna.num_prot_tokens.min())}~{int(dna.num_prot_tokens.max())}")
    print("고정 잔기 수(num_prot_tokens − num_design)",
          sorted((dna.num_prot_tokens - dna.num_design).unique()))
    print("타깃 토큰 수(num_tokens − num_prot_tokens)",
          sorted((dna.num_tokens - dna.num_prot_tokens).unique()))

명세에서 계산한 숫자와 결과 CSV 가 그대로 맞아요. 길이가 실행마다 달라지는 건 `design_insertions` 의 3~8잔기 때문이고, 모든 디자인에서 고정 잔기 수가 72로 일정한 게 `not_design` 이 작동했다는 증거예요.

여기서 안 맞는다면 `exclude`/`design`/`not_design` 범위가 겹치거나 빠진 것이니 명세부터 다시 보세요.

## 4) Zn 배위 잔기가 정말 지켜졌나 (본문 11.4)

72잔기를 왜 고정했는지가 이 챕터의 핵심이에요. zinc finger 는 `Cys-X2-Cys-X12-His-X3-His` 로 Zn²⁺ 을 붙잡는데, **이 4잔기 중 하나만 바뀌어도 Zn 이 안 붙고 finger 가 무너져요.** `zf.cif` 에는 이런 site 가 6벌 있고(2.0~2.4Å 배위), 그 24잔기가 전부 `not_design` 안에 들어 있는지 확인합니다.

In [ ]:
ZN_SITES = [(13, 16, 29, 33), (41, 44, 57, 61), (74, 77, 90, 94),
            (102, 105, 118, 122), (131, 134, 147, 151), (159, 162, 175, 179)]
for i, site in enumerate(ZN_SITES, 1):
    inside = all(r in NOT_DESIGN for r in site)
    print(f"finger {i} — Cys{site[0]},Cys{site[1]} His{site[2]},His{site[3]}"
          f" | not_design 고정 {inside}")
print(f"\nZn 배위 잔기 {sum(len(s) for s in ZN_SITES)}개 중 고정된 것",
      sum(1 for s in ZN_SITES for r in s if r in NOT_DESIGN))

if "designed_chain_sequence" in dna.columns:
    print("\n설계 서열의 Cys 개수 (6 finger x 2 = 12 이면 그대로 보존)")
    for _, r in dna.head(5).iterrows():
        s = str(r["designed_chain_sequence"])
        print(f"  rank{int(r['final_rank'])} {r['id']:15s} len={len(s):4d} Cys={s.count('C'):3d}"
              f" His={s.count('H'):3d}")

Cys 가 정확히 12개면 배위 잔기가 보존되고 새 Cys 도 안 섞인 정상 상태예요. 12보다 적으면 `not_design` 범위가 잘못된 것이고, 많으면 재설계 영역에서 자유 Cys 가 생긴 것이니 후보에서 빼거나 범위를 넓혀 다시 돌리세요. His 는 배위 12개에 재설계 영역에서 더 생길 수 있어 12개 이상이 정상이에요.

효소의 catalytic triad, 이황화 Cys, 금속 결합 부위처럼 **기능에 필수인 잔기**는 어느 타깃에서든 이렇게 고정합니다.

## 5) DNA 결합 결과 — H-bond 가 자릿수부터 다르다 (본문 11.5)

명세가 의도대로 지켜졌으니 이제 성능을 봅니다. DNA 결합에서 가장 먼저 눈에 띄는 건 인터페이스 수소결합 개수예요.

In [ ]:
dna[cols_in(dna, "id", "final_rank", "design_ptm", "design_to_target_iptm", "filter_rmsd",
             "plip_hbonds_refolded", "num_design")]

In [ ]:
rows = load_metrics(DNA_CSV)
FIG = my_fig("11_dna_metrics.png")
metrics_overview(rows, "DNA-binding (Zinc Finger) — Design Metrics Overview", FIG)
from IPython.display import Image; Image(FIG)

레퍼런스(num_designs=30) 기준 H-bond 는 20~42개, 평균 30.8개예요. 단백질-단백질 결합이 보통 한 자릿수인 것과 비교하면 차원이 다르죠 — 인산 골격의 음전하를 Arg/Lys 가 줄줄이 따라가며 붙잡기 때문이에요(본문 11.2). pTM 0.50~0.60 으로 다소 낮은 건 zinc finger 가 작고 복잡한 도메인이라 그래요.

## 6) ipTM 상한만 보면 안 되는 이유 (본문 11.5)

레퍼런스 그래프에서 ipTM 막대는 최고 0.67 까지 올라가요. 나노바디(Ch.09)나 항체(Ch.08)보다 높아서 "DNA 는 붙이기 쉽구나" 하고 넘어가기 쉬운데, **그 상한이 어느 디자인에서 나왔는지**를 반드시 같이 봐야 합니다.

In [ ]:
chk = cols_in(dna, "final_rank", "id", "design_to_target_iptm", "filter_rmsd",
              "plip_hbonds_refolded")
sub = dna[chk].sort_values("design_to_target_iptm", ascending=False) if "design_to_target_iptm" in dna.columns else dna[chk]
print(sub.to_string(index=False))
if {"design_to_target_iptm", "filter_rmsd"} <= set(dna.columns):
    good = dna[dna.filter_rmsd < 3.0]
    bad  = dna[dna.filter_rmsd >= 3.0]
    print(f"\nRMSD 3A 미만 {len(good)}개 — ipTM {good.design_to_target_iptm.min():.3f}"
          f"~{good.design_to_target_iptm.max():.3f}, RMSD {good.filter_rmsd.min():.2f}"
          f"~{good.filter_rmsd.max():.2f}A")
    if len(bad):
        print(f"RMSD 3A 이상 {len(bad)}개 — ipTM {bad.design_to_target_iptm.min():.3f}"
              f"~{bad.design_to_target_iptm.max():.3f}, RMSD {bad.filter_rmsd.min():.2f}"
              f"~{bad.filter_rmsd.max():.2f}A")
        print("  " + ", ".join(f"rank{int(r.final_rank)} {r.id}" for _, r in bad.iterrows()))

레퍼런스에서 ipTM 최고 0.670·0.635 는 rank 10(`zinc_finger_28`)·rank 9(`zinc_finger_29`)의 값이고, 이 둘은 RMSD 가 11.6Å 대로 **자기일관성이 무너진** 디자인이에요. RMSD 필터를 통과하지 못했는데도 다른 지표가 좋아 최종 10위 안에 올라온 사례죠. 자기일관성이 확보된 상위 8개만 보면 ipTM 은 0.503~0.588 이에요.

판정 기준. **최종 선별셋에 들었다는 것이 모든 지표가 좋다는 뜻은 아니에요.** 실험 후보를 고를 땐 순위만 보지 말고 RMSD 같은 자기일관성 지표를 따로 확인하고, RMSD 가 큰 디자인은 ipTM 이 아무리 높아도 제외하세요.

## 7) RNA 타깃 — 1URN 의 U1 snRNA 헤어핀 (본문 11.6)

DNA 를 했으니 RNA 로 넘어가요. 방법은 완전히 동일해요 — RNA 가 든 CIF 를 `include` 하면 2절에서 본 자동 인식이 그대로 작동합니다. 다만 이번엔 구조를 숨기지 않고(`structure_groups: "all"`) 타깃 구조를 그대로 보여줘요.

```yaml
entities:
  - protein: { id: P, sequence: 60..120 }   # RNA 에 붙을 단백질
  - file:
      path: rna_target.cif
      include:
        - chain: { id: R }    # RNA 가닥 — RNA 로 자동 인식
      structure_groups: "all"
```

커밋된 `data/rna` 는 **1URN 의 U1 snRNA 헤어핀(20 nt)** 을 RNA 만 깨끗이 추출해 단일 체인 R 로 만든 타깃에 `protein-anything`, num_designs 30, budget 10 으로 돌린 결과예요. 이 절부터는 여러분의 `my_run/`(DNA) 대신 이 레퍼런스를 씁니다.

In [ ]:
RNA_CSV = "data/rna/final_designs_metrics_10.csv"
rna = pd.read_csv(RNA_CSV).sort_values("final_rank")

rna_cif = sorted(pathlib.Path("data/rna/final_designs").glob("*.cif"))
if rna_cif:
    for l in pathlib.Path(rna_cif[0]).read_text().splitlines():
        if "polyribonucleotide" in l and l.split()[1] == "polyribonucleotide":
            seq = l.split()[-1]
            print(f"타깃 RNA {len(seq)} nt | {seq}")
if {"num_tokens", "num_prot_tokens", "num_resolved_tokens"} <= set(rna.columns):
    print("타깃 토큰 수(num_tokens − num_prot_tokens)",
          sorted((rna.num_tokens - rna.num_prot_tokens).unique()))
    print("좌표가 풀리지 않은 토큰(num_tokens − num_resolved_tokens)",
          sorted((rna.num_tokens - rna.num_resolved_tokens).unique()))
display(rna[cols_in(rna, "id", "final_rank", "design_ptm", "design_to_target_iptm",
                    "filter_rmsd", "plip_hbonds_refolded", "num_design")])

In [ ]:
rows = load_metrics(RNA_CSV)
FIG = my_fig("11_rna_metrics.png")
metrics_overview(rows, "RNA-binding (U1 snRNA hairpin) — Design Metrics Overview", FIG)
from IPython.display import Image; Image(FIG)

타깃 토큰이 20으로 일정해 헤어핀 20 nt 단일 가닥이 그대로 들어간 걸 확인할 수 있어요(그중 3개는 좌표가 풀리지 않아 결과 구조에는 17 nt 만 그려져요). ipTM 0.293~0.451, pTM 0.441~0.652, RMSD 2.16~2.47Å 로 RMSD 는 10개 모두 양호해요 — 6절 같은 자기일관성 붕괴는 없습니다.

## 8) DNA vs RNA — 같은 방법, 다른 인터페이스 (본문 11.6)

두 실행을 나란히 놓으면 타깃의 물리적 차이가 인터페이스 지표로 드러나요.

In [ ]:
dna_rows = load_metrics(DNA_CSV)
rna_rows = load_metrics(RNA_CSV)
FIG = my_fig("11_dna_rna_hbonds.png")
compare_bars({"DNA (zinc finger)": dna_rows, "RNA (hairpin)": rna_rows},
             "plip_hbonds_refolded", "DNA vs RNA — mean interface H-bonds",
             "mean H-bond count", FIG)
for tag, d in (("DNA", dna), ("RNA", rna)):
    if {"plip_hbonds_refolded", "design_to_target_iptm"} <= set(d.columns):
        print(f"{tag} — H-bond 평균 {d.plip_hbonds_refolded.mean():.1f}"
              f" (범위 {int(d.plip_hbonds_refolded.min())}~{int(d.plip_hbonds_refolded.max())})"
              f" | ipTM {d.design_to_target_iptm.min():.3f}~{d.design_to_target_iptm.max():.3f}")
    if {"ARG_fraction", "LYS_fraction"} <= set(d.columns):
        print(f"     양전하 잔기 비율(Arg+Lys) 평균 {(d.ARG_fraction + d.LYS_fraction).mean():.3f}")
from IPython.display import Image; Image(FIG)

H-bond 평균은 DNA 30.8 대 RNA 9.3 이에요. 둘 다 인산 골격이 음전하지만 타깃이 36+36 nt 이중나선이냐 20 nt 단일가닥 헤어핀이냐에 따라 접촉면이 이만큼 차이나요. 설계 서열의 Arg+Lys 비율도 DNA 쪽이 훨씬 높게 나오는데, 음전하 골격을 따라가는 정전기 결합이 실제로 서열에 반영됐다는 뜻이에요.

다만 H-bond 수가 곧 결합 품질은 아니에요. DNA 결합에서 진짜 중요한 건 **염기 서열 특이성**이고, 이건 총 개수보다 인터페이스의 기하·위치에 달려 있어요. H-bond 가 많은 후보 중 결합부위가 의도한 염기에 놓이는지를 PyMOL 로 확인하고, 비슷한 서열에 오프타깃으로 붙지 않는지 비교하세요.

## 9) 두 복합체를 눈으로 — DNA·RNA rank 1 디자인 3D (본문 11.5·11.6)

H-bond 30개와 9개의 차이가 어디서 오는지는 그림 한 장이면 끝나요.
먼저 DNA 쪽입니다 — **주황이 설계 zinc finger 단백질, 청록이 DNA 두 가닥, 보라 구가 Zn²⁺**,
그리고 **노란 stick 이 Zn 을 실제로 붙잡고 있는 잔기**(거리 3Å 안, 4)절의 그 자리)예요. 마우스로 회전·확대할 수 있어요(휠 = 줌, 드래그 = 회전). 구조가 안 보이면 `py3Dmol` 설치 로그를 확인하고 셀을 한 번 더 실행하세요.

In [ ]:
import importlib.util, sys, pathlib
for _pkg in ("py3Dmol", "gemmi"):                       # 없으면 그 자리에서 설치
    if importlib.util.find_spec(_pkg) is None:
        _run(f'"{sys.executable}" -m pip -q install {_pkg}')
import py3Dmol, gemmi

C_DESIGN  = ["#e8883a", "#c0392b"]              # 설계 사슬 — 주황·빨강
C_TARGET  = ["#3477b5", "#7f8c8d", "#8e44ad"]   # 타깃 단백질 — 파랑·회색·보라
C_NUCLEIC = "#1aa090"                           # 타깃 핵산 — 청록

def chain_seq3d(ch):
    """사슬의 한 글자 서열(폴리머 잔기만)."""
    poly = ch.get_polymer()
    return gemmi.one_letter_code([r.name for r in poly]).upper() if len(poly) else ""

def chain_kind3d(ch):
    """protein / dna / rna / hetero(리간드·금속) 판별."""
    poly = ch.get_polymer()
    if not len(poly):
        return "hetero"
    t = str(poly.check_polymer_type())
    return "dna" if "Dna" in t else "rna" if "Rna" in t else "protein"

def load_design(pattern, ref, row=None,
                seq_cols=("full_sequence_1", "full_sequence_2", "designed_chain_sequence")):
    """최종 CIF 를 찾아 열고 (pdb문자열, model, 설계사슬, 타깃사슬, 사슬→CSV컬럼) 을 돌려줍니다.
    설계 사슬은 CSV 의 설계 서열과 사슬 서열이 같은 것으로 정해요 — 사슬 라벨을 하드코딩하지 않습니다."""
    cif = find_one(pattern, ref)                        # [내 결과]/[레퍼런스] 를 스스로 출력합니다
    print("띄울 구조 :", cif, "→", "내 결과 my_run/" if "my_run" in str(cif) else "커밋된 레퍼런스 data/")
    st = gemmi.read_structure(str(cif)); st.setup_entities()
    model = st[0]

    want = {}
    if row is not None:
        for col in seq_cols:
            v = row[col] if col in getattr(row, "index", []) else None
            if isinstance(v, str) and v.strip():
                want.setdefault(v.strip().upper(), col)   # 같은 서열이면 먼저 온 컬럼을 씁니다
    match  = {ch.name: want[chain_seq3d(ch)] for ch in model if chain_seq3d(ch) in want}
    design = list(match)
    if not design:                                        # 폴백 — 설계물은 보통 가장 짧은 단백질 사슬
        prot = [ch for ch in model if chain_kind3d(ch) == "protein"]
        design = [min(prot, key=lambda c: len(c.get_polymer())).name] if prot else []
        print("  (CSV 서열과 일치하는 사슬이 없어 가장 짧은 단백질 사슬을 설계물로 봅니다)")
    target = [ch.name for ch in model if ch.name not in design]

    for ch in model:
        n = len(ch.get_polymer()) or len(ch)
        # hetero(리간드·금속)는 one-letter 서열이 안 나와 칸이 비어 보여요 — 잔기 이름을 대신 보여줍니다.
        what = chain_seq3d(ch)[:28] or "·".join(sorted({r.name for r in ch}))
        print(f"  사슬 {ch.name:<2s} {'설계' if ch.name in design else '타깃'} "
              f"| {chain_kind3d(ch):<7s} {n:>4d} | {what}")
    return st.make_pdb_string(), model, design, target, match

def complex_view(pdb, model, design, target, width=760, height=540):
    """설계 사슬(주황)·타깃 단백질(파랑)·핵산(청록) cartoon + 비폴리머(리간드 stick·이온 sphere)."""
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb, "pdb")
    view.setStyle({}, {"cartoon": {"color": "#cfd6dd"}})
    ti = 0
    for ch in model:
        kind = chain_kind3d(ch)
        if kind == "hetero":
            continue
        if ch.name in design:
            color = C_DESIGN[design.index(ch.name) % len(C_DESIGN)]
        elif kind in ("dna", "rna"):
            color = C_NUCLEIC
        else:
            color = C_TARGET[ti % len(C_TARGET)]; ti += 1
        style = {"cartoon": {"color": color}}
        if kind in ("dna", "rna"):
            style["stick"] = {"color": color, "radius": 0.12}   # 염기까지 보이게
        view.setStyle({"chain": ch.name}, style)
    for name, natoms in {r.name: len(r) for ch in model for r in ch if r.het_flag == "H"}.items():
        if natoms == 1:                                   # 금속·이온
            view.addStyle({"resn": name}, {"sphere": {"color": "#6c5ce7", "radius": 0.9}})
        else:                                             # 리간드
            view.addStyle({"resn": name}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})
    view.setBackgroundColor("white")
    return view

def designed_segments(full, des, min_len=3):
    """이어붙여 저장된 재설계 구간(des)을 전체 서열(full) 위에 되짚어 (시작,끝) 1-based 목록으로.
    CDR 번호를 노트북에 적어 두지 않고 결과 CSV 에서 되찾기 위한 것입니다."""
    full, des = str(full).upper(), str(des).upper()
    out, i, pos = [], 0, 0
    while i < len(des):
        best, at = 0, -1
        for L in range(len(des) - i, min_len - 1, -1):
            j = full.find(des[i:i + L], pos)
            if j >= 0:
                best, at = L, j
                break
        if best == 0:
            i += 1
            continue
        out.append((at + 1, at + best))
        i += best; pos = at + best
    return out

def seg_resi(model, chain_id, segs):
    """서열 위 구간 → 그 사슬의 실제 잔기 번호 목록(3Dmol 의 resi 선택용)."""
    ch = {c.name: c for c in model}[chain_id]
    nums = [r.seqid.num for r in ch.get_polymer()]
    return [n for a, b in segs for n in nums[a - 1:b]]

# 사슬을 찾아준 컬럼 → 그 사슬의 '재설계 구간' 컬럼
DES_COL = {"full_sequence_1": "designed_sequence_1", "full_sequence_2": "designed_sequence_2",
           "designed_chain_sequence": "designed_sequence"}

dna_top = dna.sort_values("final_rank").iloc[0]
print("DNA rank 1 디자인 :", dna_top["id"])
pdb, model, DESIGN, TARGET, MATCH = load_design(
    f"final*designs/*{dna_top['id']}*.cif", "data/dna", row=dna_top)

view = complex_view(pdb, model, DESIGN, TARGET)

# Zn 배위 잔기 — 좌표에서 직접 찾습니다(잔기 번호를 적어 두지 않아요)
zn_pos = [a.pos for ch in model for r in ch if r.name == "ZN" for a in r]
for ch in model:
    if ch.name not in DESIGN:
        continue
    coord = [r.seqid.num for r in ch if r.name in ("CYS", "HIS")
             and any(a.pos.dist(p) <= 3.0 for a in r for p in zn_pos)]
    view.addStyle({"chain": ch.name, "resi": coord},
                  {"stick": {"colorscheme": "yellowCarbon", "radius": 0.2}})
    print(f"  Zn 이온 {len(zn_pos)}개 | 사슬 {ch.name} 의 배위 잔기 {len(coord)}개 @ {'·'.join(map(str, coord))}")

view.zoomTo()
view.show()

print("\n무엇을 봐야 하나")
print("  · 주황 단백질의 손가락(finger)들이 청록 이중나선의 큰 홈(major groove)에 차례로 꽂혀 있으면 정상이에요.")
print("  · 보라 Zn 구마다 노란 stick 4개(Cys 2 + His 2)가 사면체로 감싸고 있으면 4)절의 고정이 지켜진 겁니다.")
print("  · 단백질이 DNA 옆에 떠 있거나 finger 가 풀려 있으면 6)절의 RMSD 큰 디자인이에요 — 후보에서 빼세요.")
if {"plip_hbonds_refolded", "filter_rmsd"} <= set(dna.columns):
    print(f"\n이 디자인의 숫자 — H-bond {int(dna_top['plip_hbonds_refolded'])}개 / "
          f"RMSD {dna_top['filter_rmsd']:.2f} A")

이번엔 RNA 쪽이에요. 같은 색 규칙(**주황 = 설계 단백질, 청록 = RNA**)으로 띄우면
타깃 크기 차이가 그대로 보입니다 — 36+36 nt 이중나선 vs 20 nt 헤어핀. 마우스로 회전·확대할 수 있어요(휠 = 줌, 드래그 = 회전). 구조가 안 보이면 `py3Dmol` 설치 로그를 확인하고 셀을 한 번 더 실행하세요.

In [ ]:
rna_top = rna.sort_values("final_rank").iloc[0]
print("RNA rank 1 디자인 :", rna_top["id"])
pdb_r, model_r, DESIGN_R, TARGET_R, MATCH_R = load_design(
    f"final*designs/*{rna_top['id']}*.cif", "data/rna", row=rna_top)

view = complex_view(pdb_r, model_r, DESIGN_R, TARGET_R)
view.zoomTo()
view.show()

print("\n무엇을 봐야 하나")
print("  · 청록 RNA 헤어핀(줄기-고리)이 주황 단백질의 β-sheet 면에 얹혀 있으면 정상이에요 (RRM 도메인의 전형적 결합).")
print("  · 접촉면이 DNA 쪽보다 눈에 띄게 좁죠 — 8)절의 H-bond 평균 30.8 대 9.3 이 이 그림입니다.")
print("  · 좌표가 풀리지 않은 nt 는 아예 안 그려져요(7)절 20 nt 중 17 nt 만 표시).")

## 10) 이 챕터에서 확인한 것 (본문 11.7)

핵산 타깃은 CIF 에 있으면 자동 인식되어 별도 프로토콜이 필요 없고(2절), 대신 **명세로 무엇을 고정할지**가 설계의 승부처였어요. `exclude`·`design_insertions`·`visibility 0`·`design`/`not_design`·`reset_res_index` 로 재설계 범위를 짜고(3절), Zn 배위 24잔기를 고정해 finger 가 무너지지 않게 했죠(4절). 결과는 DNA 가 H-bond·ipTM 모두 높지만 순위 상단에 자기일관성이 깨진 디자인이 섞일 수 있고(6절), RNA 는 접촉면이 작아 지표가 전반적으로 낮게 나왔어요(7·8절).

이걸로 Part B 의 타깃별 실습을 모두 마쳤어요. 단백질·펩타이드·항체·나노바디·소분자·핵산 어느 타깃이든 **명세로 제약을 걸고 → 실행하고 → 여러 지표를 따로 확인해 후보를 고르는** 흐름은 같습니다. 이제 여러분의 타깃으로 같은 흐름을 돌려보세요.